In [ ]:
import kagglehub

import pandas as pd
import torch
import torch.nn as nn
from torch.optim import AdamW
import matplotlib.pyplot as plt
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split


# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors
import torch
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
import numpy as np

# 1. Convert NumPy arrays to PyTorch Tensors
categorical_cols = labels_df.select_dtypes(include=["object"]).columns
for col in categorical_cols:
    print(f"Encoding column: {col}")
    le = LabelEncoder()
    labels_df[col] = le.fit_transform(labels_df[col])
labels_df
numerical_cols = labels_df.select_dtypes(exclude='object')

X_train_tensor = torch.from_numpy(X_train).float()
y_train_tensor = torch.from_numpy(y_train).long()
X_test_tensor = torch.from_numpy(X_test).float()    # we need to ensure the data types are correct
y_test_tensor = torch.from_numpy(y_test).long()



In [ ]:
# 2. Create TensorDataset objects


train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)


In [ ]:
# 3. Create DataLoaders


batch_size = 32
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
# 4. Print shape of one batch
data_iter = iter(train_loader)
images, labels = next(data_iter)

print(f"images shape: {images.shape}") # Expect: [32, Channels, Height, Width]
print(f"labels shape: {labels.shape}") # Expect: [32]


In [ ]:
# 5. Display sample images
def show_images(imgs, labels, num_images=5):
    plt.figure(figsize=(12, 4))
    for i in range(num_images):
        plt.subplot(1, num_images, i + 1)


        img = imgs[i].numpy()
        if img.ndim == 3:
            img = np.transpose(img, (1, 2, 0))

        plt.imshow(img, cmap='gray' if img.ndim == 2 else None)
        plt.title(f"label: {labels[i].item()}")
        plt.axis('off')
    plt.show()


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class AgePredictor(nn.Module):
    def __init__(self, input_size, num_classes):
        super(AgePredictor, self).__init__()
        # Flatten the (C, H, W) image into a single vector
        self.flatten = nn.Flatten()

        # 4 Linear Layers
        self.fc1 = nn.Linear(input_size, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, num_classes) # num_classes = number of age categories

    def forward(self, x):
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = self.fc4(x) # No activation on the final layer (CrossEntropyLoss handles it)
        return x

In [ ]:
def train_loop(model, loader, loss_fn, optimizer, device):
    model.train()
    running_loss = 0.0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        # Forward pass
        outputs = model(images)
        loss = loss_fn(outputs, labels)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
    return running_loss / len(loader)



In [ ]:
# Task 3: Write your validation loop here:
def validate_loop(model, loader, loss_fn, device):
    model.eval()
    running_loss = 0.0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = loss_fn(outputs, labels)
            running_loss += loss.item()
    return running_loss / len(loader)

In [ ]:
# Task 4: Define device, model, loss, optimizer:

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Model parameters
input_dim = X_train.shape[1]   # Number of tabular features
hidden_dim = 14                # Design choice
output_dim = 4                 # Weather classes: Rainy, Sunny, Cloudy, Snowy

# Instantiate model
model = NN4Layer(input_dim, hidden_dim, output_dim).to(device)

# Print the model architecture
print("Model Architecture:\n")
print(model)

# Calculate the total number of trainable parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal trainable parameters: {total_params}")

In [ ]:
# Task 5: Start training for 20 epochs:


sample_batch = next(iter(train_loader))[0]
c, h, w = sample_batch.shape[1], sample_batch.shape[2], sample_batch.shape[3]
flattened_size = c * h * w

print(f"Correct Input Size: {flattened_size}")

# 2. Update the Model Class
class AgePredictor(nn.Module):
    def __init__(self, input_size, num_classes):
        super(AgePredictor, self).__init__()
        self.flatten = nn.Flatten()


        self.fc1 = nn.Linear(input_size, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = self.fc4(x)
        return x

# 3. Re-initialize
model = AgePredictor(flattened_size, num_age_classes).to(device)
# Define Device

# 4. Start Training for 20 Epochs

train_losses = []
val_losses = []
epochs = 20

print(f"Starting training on {device}...")

for epoch in range(epochs):
    train_loss = train_loop(model, train_loader, loss_fn, optimizer, device)
    val_loss = validate_loop(model, test_loader, loss_fn, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

In [ ]:
# Task 1: Write your code here:
# Plotting results
plt.figure(figsize=(7, 5))

plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Task 2 (Bonus): Write your code here: